# Clip thật cho ba kiểu lỗi đổi ID (slide 10)

Notebook riêng, chỉ cần **CPU** (Runtime → Change runtime type → CPU), không tốn giờ GPU.

Cùng thiết lập với 473 lỗi trong báo cáo: ByteTrack trên phát hiện oracle (hộp lấy từ nhãn). Script tự chọn các lỗi dễ nhìn rồi cắt clip quanh lỗi:

- xanh: track trước lỗi (`ID a`), cam: track sau lỗi (`ID b`), xám: các track khác
- khung trắng nét đứt: vật lúc không có track nào bám
- nhãn đỏ `đổi ID a→b` từ khung xảy ra lỗi; clip phát chậm 2 lần

Chỉ tải nhãn (vài MB) và đúng các khung cần cắt (đọc theo đoạn từ zip trên Hugging Face), không tải cả bộ dữ liệu.

**Trước khi chạy:** push mã mới nhất lên GitHub (`scripts/export_switch_examples.py` phải có trong repo).

## 1. Chuẩn bị

In [ ]:
TYPES = 'fragment swap takeover'   # ví dụ chỉ cắt lại hai kiểu: 'fragment takeover'
!python scripts/export_switch_examples.py --data $DATA --remote --out $OUT --per-type 3 --types $TYPES

## 2. Đăng nhập Hugging Face

Cần để đọc khung hình từ zip của FastTracker. Nếu đã lưu `HF_TOKEN` trong mục Secrets của Colab thì ô này không hỏi gì.

In [ ]:
from huggingface_hub import get_token, login
if not get_token():
    login()
print('token:', 'có' if get_token() else 'chưa có')

## 3. Tải nhãn (gt.txt và seqinfo.ini của 12 video, vài MB)

In [ ]:
!python scripts/download_fasttracker.py --out $DS --mode gt
print(sorted(os.listdir(DATA)))

## 4. Chọn lỗi và cắt clip

Mặc định 3 clip mỗi kiểu, mỗi clip 1,5 giây trước và sau lỗi. Thời gian chạy chủ yếu là tải khung hình (khoảng 90 khung mỗi clip).

Script bỏ các lỗi xảy ra khi vật chạm mép ảnh (xe đang ra hoặc vào khung hình), vì đó là ví dụ kém cho cơ chế lỗi, và in ra số lỗi như vậy theo từng kiểu. Với đứt mạch và chiếm ID, lỗi có vật bị che ngay trước đó được ưu tiên.

Tuỳ chọn: `TYPES` để chỉ cắt một số kiểu, `--per-type 5` để có nhiều lựa chọn hơn, `--min-h 80` để chỉ lấy vật lớn, `--pre 2 --post 2` để clip dài hơn, `--seqs task_day_left_turn ...` để giới hạn video, `--allow-border` để lấy cả lỗi ở mép ảnh.

In [ ]:
TYPES = 'fragment swap takeover'   # ví dụ chỉ cắt lại hai kiểu: 'fragment takeover'
!python scripts/export_switch_examples.py --data $DATA --remote --out $OUT --per-type 3 --types $TYPES

## 5. Xem và chọn

In [ ]:
import pandas as pd
from IPython.display import Video, Markdown, display
ex = pd.read_csv(f'{OUT}/examples.csv')
display(ex[['clip', 'seq', 'frame', 'prev_trk', 'new_trk', 'box_h', 'gt_vis', 'n_gt_overlap']])
for r in ex.itertuples():
    display(Markdown(f'**{r.clip}**: {r.seq}, khung {r.frame}, ID {r.prev_trk}→{r.new_trk}'))
    display(Video(f'{OUT}/{r.clip}', embed=True, width=640))

## 6. Tải các clip đã chọn về máy

Sửa số thứ tự theo clip bạn chọn ở trên (ví dụ `'swap': 2` là `swap_2.mp4`; bỏ các kiểu không cần), chạy ô, rồi đính kèm các tệp `fragment.mp4`, `swap.mp4`, `takeover.mp4` vào chat. Tệp cũng được chép vào `switch_examples/chosen` trên Drive.

Lưu ý: một "lỗi" vẫn có thể là lỗi nhãn (ID trong nhãn nhảy sang xe khác). Chọn clip mà bằng mắt thấy rõ tracker sai.

In [ ]:
import shutil
from google.colab import files
CHOSEN = {'fragment': 1, 'swap': 1, 'takeover': 1}
os.makedirs(f'{OUT}/chosen', exist_ok=True)
for typ, k in CHOSEN.items():
    for ext in ('mp4', 'png'):
        shutil.copy(f'{OUT}/{typ}_{k}.{ext}', f'{OUT}/chosen/{typ}.{ext}')
    files.download(f'{OUT}/chosen/{typ}.mp4')